###for linux


print('Installing Icarus Verilog (iverilog)...')

!sudo apt-get update

!sudo apt-get install -y iverilog

print('Icarus Verilog (iverilog) installed successfully!')

!iverilog -V

In [3]:
import os
import shutil
import subprocess
from pathlib import Path

compiler = shutil.which("iverilog")
local_compiler = Path.home() / "iverilog" / "bin" / ("iverilog.exe" if os.name == "nt" else "iverilog")
if compiler is None and local_compiler.exists():
    compiler = str(local_compiler)

if compiler is None and os.name == "nt" and shutil.which("choco"):
    print("Icarus Verilog is missing; installing it with Chocolatey...")
    subprocess.run(
        ["choco", "install", "iverilog", "--exact", "--yes", "--no-progress"],
        check=True,
    )
    compiler = shutil.which("iverilog")

if compiler is None:
    raise RuntimeError(
        "Icarus Verilog (iverilog) is required. Install it with "
        "'choco install iverilog -y' on Windows or "
        "'sudo apt-get install -y iverilog' on Linux."
    )

print(f"Icarus Verilog found at: {compiler}")
subprocess.run([compiler, "-V"], check=True)

Icarus Verilog found at: C:\Users\satya\iverilog\bin\iverilog.exe


CompletedProcess(args=['C:\\Users\\satya\\iverilog\\bin\\iverilog.exe', '-V'], returncode=0)

In [4]:
print('Installing pyVCD (Python VCD parser)...')
!pip install pyvcd

print('pyVCD installed successfully!')

Installing pyVCD (Python VCD parser)...
pyVCD installed successfully!



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%%writefile load_analyzer.v
module load_analyzer (
    input  [15:0] pred_it_power,
    input  [15:0] pred_cooling_power,
    input  [15:0] pred_hvac_power,
    input  [15:0] pred_pump_power,

    output [15:0] predicted_power_total,
    output [2:0]  predicted_zone,
    output        predicted_high_power,
    output        reserve_trigger,
    output        cooling_boost
);

localparam [2:0] ZONE_EMERGENCY = 3'b000;
localparam [2:0] ZONE_BUFFER    = 3'b001;
localparam [2:0] ZONE_LOW       = 3'b010;
localparam [2:0] ZONE_MEDIUM    = 3'b011;
localparam [2:0] ZONE_HIGH      = 3'b100;
localparam [2:0] ZONE_CRITICAL  = 3'b101;

// 1000 W = 100% normalized capacity.
assign predicted_power_total =
    pred_it_power +
    pred_cooling_power +
    pred_hvac_power +
    pred_pump_power;

function [2:0] classify;
    input [15:0] p;
    begin
        if      (p < 16'd300) classify = ZONE_EMERGENCY;
        else if (p < 16'd400) classify = ZONE_BUFFER;
        else if (p < 16'd550) classify = ZONE_LOW;
        else if (p < 16'd750) classify = ZONE_MEDIUM;
        else if (p < 16'd850) classify = ZONE_HIGH;
        else                  classify = ZONE_CRITICAL;
    end
endfunction

assign predicted_zone = classify(predicted_power_total);

assign predicted_high_power =
    (predicted_zone == ZONE_HIGH) ||
    (predicted_zone == ZONE_CRITICAL);

assign reserve_trigger = (predicted_zone == ZONE_EMERGENCY);
assign cooling_boost   = (predicted_zone == ZONE_CRITICAL);

endmodule

Writing load_analyzer.v


In [6]:
%%writefile load_analyzer_top.v
module load_analyzer_top (
    input  [15:0] s1_pred_it, s1_pred_cooling, s1_pred_hvac, s1_pred_pump,
    input  [15:0] s2_pred_it, s2_pred_cooling, s2_pred_hvac, s2_pred_pump,
    input  [15:0] s3_pred_it, s3_pred_cooling, s3_pred_hvac, s3_pred_pump,

    output [15:0] s1_pred_total, s2_pred_total, s3_pred_total,
    output [2:0]  s1_pred_zone, s2_pred_zone, s3_pred_zone,
    output        s1_pred_high, s2_pred_high, s3_pred_high,
    output        s1_reserve_trigger, s2_reserve_trigger, s3_reserve_trigger,
    output        s1_cooling_boost, s2_cooling_boost, s3_cooling_boost
);

    load_analyzer LA_S1 (
        .pred_it_power(s1_pred_it),
        .pred_cooling_power(s1_pred_cooling),
        .pred_hvac_power(s1_pred_hvac),
        .pred_pump_power(s1_pred_pump),
        .predicted_power_total(s1_pred_total),
        .predicted_zone(s1_pred_zone),
        .predicted_high_power(s1_pred_high),
        .reserve_trigger(s1_reserve_trigger),
        .cooling_boost(s1_cooling_boost)
    );

    load_analyzer LA_S2 (
        .pred_it_power(s2_pred_it),
        .pred_cooling_power(s2_pred_cooling),
        .pred_hvac_power(s2_pred_hvac),
        .pred_pump_power(s2_pred_pump),
        .predicted_power_total(s2_pred_total),
        .predicted_zone(s2_pred_zone),
        .predicted_high_power(s2_pred_high),
        .reserve_trigger(s2_reserve_trigger),
        .cooling_boost(s2_cooling_boost)
    );

    load_analyzer LA_S3 (
        .pred_it_power(s3_pred_it),
        .pred_cooling_power(s3_pred_cooling),
        .pred_hvac_power(s3_pred_hvac),
        .pred_pump_power(s3_pred_pump),
        .predicted_power_total(s3_pred_total),
        .predicted_zone(s3_pred_zone),
        .predicted_high_power(s3_pred_high),
        .reserve_trigger(s3_reserve_trigger),
        .cooling_boost(s3_cooling_boost)
    );

endmodule

Writing load_analyzer_top.v


In [7]:
%%writefile health_maintenance.v
module health_maintenance (
    input  [15:0] predicted_it_power,
    input  [15:0] predicted_cooling_power,
    input  [15:0] predicted_hvac_power,
    input  [7:0]  predicted_temperature,
    input  [15:0] reserve_cooling_boost,
    input  [1:0]  maintenance_risk,

    output [7:0]  effective_temperature,
    output [1:0]  thermal_status,
    output [1:0]  predicted_thermal_status,
    output        thermal_safe,
    output        predicted_thermal_safe,
    output        thermal_redistribution_trigger,
    output        cooling_malfunction,
    output        cooling_reserve_request,
    output        maintenance_ok,
    output        health_eligible
);
    localparam [1:0] THERMAL_SAFE     = 2'b00;
    localparam [1:0] THERMAL_WARNING  = 2'b01;
    localparam [1:0] THERMAL_CRITICAL = 2'b10;

    localparam [1:0] MAINT_NORMAL   = 2'b00;
    localparam [1:0] MAINT_WATCH    = 2'b01;
    localparam [1:0] MAINT_WARNING  = 2'b10;
    localparam [1:0] MAINT_CRITICAL = 2'b11;

    localparam [7:0] TEMP_WARNING  = 8'd70;
    localparam [7:0] TEMP_CRITICAL = 8'd85;

    // Simple software-modelled reserve cooling effect:
    // every 20 W of reserve cooling represents 1 C reduction.
    wire [15:0] temp_reduction = reserve_cooling_boost / 16'd20;
    wire signed [16:0] temp_after_boost =
        $signed({1'b0, predicted_temperature}) - $signed({1'b0, temp_reduction});

    assign effective_temperature =
        (temp_after_boost <= 0) ? 8'd0 :
        (temp_after_boost >= 100) ? 8'd100 : temp_after_boost[7:0];

    function [1:0] classify_thermal;
        input [7:0] t;
        begin
            if      (t < TEMP_WARNING) classify_thermal = THERMAL_SAFE;
            else if (t < TEMP_CRITICAL) classify_thermal = THERMAL_WARNING;
            else                        classify_thermal = THERMAL_CRITICAL;
        end
    endfunction

    assign predicted_thermal_status = classify_thermal(predicted_temperature);
    assign thermal_status           = classify_thermal(effective_temperature);

    assign predicted_thermal_safe = (predicted_thermal_status != THERMAL_CRITICAL);
    assign thermal_safe           = (thermal_status != THERMAL_CRITICAL);

    assign thermal_redistribution_trigger =
        (thermal_status == THERMAL_WARNING) ||
        (thermal_status == THERMAL_CRITICAL);

    // Predictive cooling malfunction check.
    // Only predicted values are used.
    assign cooling_malfunction =
        ((3 * (predicted_cooling_power + predicted_hvac_power)) < 16'd80) &&
        (predicted_it_power > 16'd600);

    assign cooling_reserve_request =
        cooling_malfunction || (thermal_status == THERMAL_CRITICAL);

    assign maintenance_ok =
        ((maintenance_risk == MAINT_NORMAL) ||
         (maintenance_risk == MAINT_WATCH)) &&
        !cooling_malfunction;

    assign health_eligible = thermal_safe && maintenance_ok;
endmodule

Writing health_maintenance.v


In [8]:
%%writefile health_maintenance_top.v
module health_maintenance_top (
    input clk,
    input rst,

    input [15:0] s1_pred_it, s2_pred_it, s3_pred_it,
    input [15:0] s1_pred_cooling, s2_pred_cooling, s3_pred_cooling,
    input [15:0] s1_pred_hvac, s2_pred_hvac, s3_pred_hvac,
    input [7:0]  s1_pred_temp, s2_pred_temp, s3_pred_temp,
    input [1:0]  s1_maint, s2_maint, s3_maint,

    input [15:0] s1_power, s2_power, s3_power,
    input        s1_reserve_trigger_la, s2_reserve_trigger_la, s3_reserve_trigger_la,
    input        s1_cooling_boost_la, s2_cooling_boost_la, s3_cooling_boost_la,

    input [1:0] source_server,
    input       emergency_power_restriction_trigger,

    output [7:0] s1_effective_temp, s2_effective_temp, s3_effective_temp,
    output [1:0] s1_thermal_status, s2_thermal_status, s3_thermal_status,
    output [1:0] s1_pred_thermal_status, s2_pred_thermal_status, s3_pred_thermal_status,
    output       s1_thermal_safe, s2_thermal_safe, s3_thermal_safe,
    output       s1_pred_thermal_safe, s2_pred_thermal_safe, s3_pred_thermal_safe,
    output       s1_thermal_redist_trigger, s2_thermal_redist_trigger, s3_thermal_redist_trigger,
    output       s1_cooling_malfunction, s2_cooling_malfunction, s3_cooling_malfunction,
    output       s1_maintenance_ok, s2_maintenance_ok, s3_maintenance_ok,
    output       s1_health_eligible, s2_health_eligible, s3_health_eligible,

    output [15:0] reserve_cooling_injection,
    output [15:0] reserve_charge_level,
    output [15:0] s1_reserve_injection, s2_reserve_injection, s3_reserve_injection,
    output        cooling_reserve_active
);

    wire s1_cooling_reserve_request, s2_cooling_reserve_request, s3_cooling_reserve_request;

    wire cooling_boost_required_combined =
        s1_cooling_boost_la || s2_cooling_boost_la || s3_cooling_boost_la ||
        s1_cooling_reserve_request || s2_cooling_reserve_request || s3_cooling_reserve_request;

    health_maintenance HM_S1 (
        .predicted_it_power(s1_pred_it),
        .predicted_cooling_power(s1_pred_cooling),
        .predicted_hvac_power(s1_pred_hvac),
        .predicted_temperature(s1_pred_temp),
        .reserve_cooling_boost(reserve_cooling_injection),
        .maintenance_risk(s1_maint),
        .effective_temperature(s1_effective_temp),
        .thermal_status(s1_thermal_status),
        .predicted_thermal_status(s1_pred_thermal_status),
        .thermal_safe(s1_thermal_safe),
        .predicted_thermal_safe(s1_pred_thermal_safe),
        .thermal_redistribution_trigger(s1_thermal_redist_trigger),
        .cooling_malfunction(s1_cooling_malfunction),
        .cooling_reserve_request(s1_cooling_reserve_request),
        .maintenance_ok(s1_maintenance_ok),
        .health_eligible(s1_health_eligible)
    );

    health_maintenance HM_S2 (
        .predicted_it_power(s2_pred_it),
        .predicted_cooling_power(s2_pred_cooling),
        .predicted_hvac_power(s2_pred_hvac),
        .predicted_temperature(s2_pred_temp),
        .reserve_cooling_boost(reserve_cooling_injection),
        .maintenance_risk(s2_maint),
        .effective_temperature(s2_effective_temp),
        .thermal_status(s2_thermal_status),
        .predicted_thermal_status(s2_pred_thermal_status),
        .thermal_safe(s2_thermal_safe),
        .predicted_thermal_safe(s2_pred_thermal_safe),
        .thermal_redistribution_trigger(s2_thermal_redist_trigger),
        .cooling_malfunction(s2_cooling_malfunction),
        .cooling_reserve_request(s2_cooling_reserve_request),
        .maintenance_ok(s2_maintenance_ok),
        .health_eligible(s2_health_eligible)
    );

    health_maintenance HM_S3 (
        .predicted_it_power(s3_pred_it),
        .predicted_cooling_power(s3_pred_cooling),
        .predicted_hvac_power(s3_pred_hvac),
        .predicted_temperature(s3_pred_temp),
        .reserve_cooling_boost(reserve_cooling_injection),
        .maintenance_risk(s3_maint),
        .effective_temperature(s3_effective_temp),
        .thermal_status(s3_thermal_status),
        .predicted_thermal_status(s3_pred_thermal_status),
        .thermal_safe(s3_thermal_safe),
        .predicted_thermal_safe(s3_pred_thermal_safe),
        .thermal_redistribution_trigger(s3_thermal_redist_trigger),
        .cooling_malfunction(s3_cooling_malfunction),
        .cooling_reserve_request(s3_cooling_reserve_request),
        .maintenance_ok(s3_maintenance_ok),
        .health_eligible(s3_health_eligible)
    );

    reserve_power_supply RESERVE (
        .s1_power(s1_power),
        .s2_power(s2_power),
        .s3_power(s3_power),
        .s1_reserve_trigger(s1_reserve_trigger_la),
        .s2_reserve_trigger(s2_reserve_trigger_la),
        .s3_reserve_trigger(s3_reserve_trigger_la),
        .source_server(source_server),
        .emergency_power_restriction_trigger(emergency_power_restriction_trigger),
        .cooling_boost_required(cooling_boost_required_combined),
        .clk(clk),
        .rst(rst),
        .s1_reserve_injection(s1_reserve_injection),
        .s2_reserve_injection(s2_reserve_injection),
        .s3_reserve_injection(s3_reserve_injection),
        .reserve_cooling_injection(reserve_cooling_injection),
        .reserve_charge_level(reserve_charge_level)
    );

    assign cooling_reserve_active = (reserve_cooling_injection != 16'd0);
endmodule

Writing health_maintenance_top.v


In [9]:
%%writefile reserve_power_supply.v
module reserve_power_supply (
    input [15:0] s1_power, s2_power, s3_power,
    input        s1_reserve_trigger, s2_reserve_trigger, s3_reserve_trigger,
    input [1:0]  source_server,
    input        emergency_power_restriction_trigger,
    input        cooling_boost_required,
    input        clk,
    input        rst,
    output reg [15:0] s1_reserve_injection, s2_reserve_injection, s3_reserve_injection,
    output reg [15:0] reserve_cooling_injection,
    output reg [15:0] reserve_charge_level
);
    localparam [15:0] RAMP_STEP          = 16'd15;
    localparam [15:0] RESTRICTION_TARGET = 16'd750;
    localparam [15:0] RESERVE_MAX        = 16'd10000;
    localparam [1:0]  NONE = 2'b00, S1 = 2'b01, S2 = 2'b10, S3 = 2'b11;

    wire [15:0] restrict_source_power =
        (source_server == S1) ? s1_power :
        (source_server == S2) ? s2_power :
        (source_server == S3) ? s3_power : 16'd0;

    wire [15:0] this_frame_cut =
        (emergency_power_restriction_trigger && (restrict_source_power > RESTRICTION_TARGET))
        ? (((restrict_source_power - RESTRICTION_TARGET) > RAMP_STEP)
            ? RAMP_STEP
            : (restrict_source_power - RESTRICTION_TARGET))
        : 16'd0;

    wire [15:0] s1_want = s1_reserve_trigger ? RAMP_STEP : 16'd0;
    wire [15:0] s2_want = s2_reserve_trigger ? RAMP_STEP : 16'd0;
    wire [15:0] s3_want = s3_reserve_trigger ? RAMP_STEP : 16'd0;
    wire [15:0] cooling_want = cooling_boost_required ? RAMP_STEP : 16'd0;

    reg [15:0] s1_inj_c, s2_inj_c, s3_inj_c, cool_inj_c, remaining;

    always @(*) begin
        remaining = reserve_charge_level;

        s1_inj_c = (remaining >= s1_want) ? s1_want : remaining;
        remaining = remaining - s1_inj_c;

        s2_inj_c = (remaining >= s2_want) ? s2_want : remaining;
        remaining = remaining - s2_inj_c;

        s3_inj_c = (remaining >= s3_want) ? s3_want : remaining;
        remaining = remaining - s3_inj_c;

        cool_inj_c = (remaining >= cooling_want) ? cooling_want : remaining;
        remaining = remaining - cool_inj_c;
    end

    // FIX: candidate next charge value is computed fresh (blocking, into a
    // 17-bit temp reg) each cycle, then clamped BEFORE it is ever written to
    // reserve_charge_level. The previous version's clamp read the stale
    // (pre-cycle) registered value via a non-blocking-assignment race, so it
    // never actually fired -- this_frame_cut (restriction recharge) could
    // push reserve_charge_level above RESERVE_MAX with nothing to bring it
    // back down, which would then make safety_check's reserve_valid (and
    // therefore final_valid) incorrectly stay low for the rest of the run.
    // Injections are already capped against `remaining` above, so the
    // subtraction side can never underflow -- only the addition side needed
    // the fix.
    reg [16:0] charge_candidate;

    always @(posedge clk or posedge rst) begin
        if (rst) begin
            s1_reserve_injection  <= 16'd0;
            s2_reserve_injection  <= 16'd0;
            s3_reserve_injection  <= 16'd0;
            reserve_cooling_injection <= 16'd0;
            reserve_charge_level  <= RESERVE_MAX;
        end else begin
            s1_reserve_injection <= s1_inj_c;
            s2_reserve_injection <= s2_inj_c;
            s3_reserve_injection <= s3_inj_c;
            reserve_cooling_injection <= cool_inj_c;

            charge_candidate = {1'b0, reserve_charge_level} + {1'b0, this_frame_cut}
                              - {1'b0, s1_inj_c} - {1'b0, s2_inj_c}
                              - {1'b0, s3_inj_c} - {1'b0, cool_inj_c};

            reserve_charge_level <= (charge_candidate > {1'b0, RESERVE_MAX})
                                     ? RESERVE_MAX
                                     : charge_candidate[15:0];
        end
    end
endmodule

Writing reserve_power_supply.v


In [10]:
%%writefile priority_eligibility.v
module priority_eligibility (
    input  [15:0] s1_pred_power, s2_pred_power, s3_pred_power,
    input         s1_health_eligible, s2_health_eligible, s3_health_eligible,
    input  [7:0]   s1_pred_temp, s2_pred_temp, s3_pred_temp,
    input  [1:0]   mode_select,

    output        s1_receiver_eligible, s2_receiver_eligible, s3_receiver_eligible,
    output [1:0]  best_receiver
);
    localparam [1:0] NONE = 2'b00;
    localparam [1:0] S1   = 2'b01;
    localparam [1:0] S2   = 2'b10;
    localparam [1:0] S3   = 2'b11;

    localparam [15:0] RECEIVER_LIMIT = 16'd600;
    localparam [7:0]  TEMP_LIMIT     = 8'd70;

    assign s1_receiver_eligible =
        (s1_pred_power < RECEIVER_LIMIT) &&
        (s1_pred_temp < TEMP_LIMIT) &&
        s1_health_eligible;

    assign s2_receiver_eligible =
        (s2_pred_power < RECEIVER_LIMIT) &&
        (s2_pred_temp < TEMP_LIMIT) &&
        s2_health_eligible;

    assign s3_receiver_eligible =
        (s3_pred_power < RECEIVER_LIMIT) &&
        (s3_pred_temp < TEMP_LIMIT) &&
        s3_health_eligible;

    // mode 00 = maximum headroom (lowest power wins)
    // mode 01 = coolest eligible server wins
    // mode 10 = fixed S1 > S2 > S3 priority
    // mode 11 = fixed S3 > S2 > S1 priority
    wire [15:0] s1_headroom_score = s1_receiver_eligible ? (16'd600 - s1_pred_power) : 16'd0;
    wire [15:0] s2_headroom_score = s2_receiver_eligible ? (16'd600 - s2_pred_power) : 16'd0;
    wire [15:0] s3_headroom_score = s3_receiver_eligible ? (16'd600 - s3_pred_power) : 16'd0;

    function [1:0] pick_headroom;
        input [15:0] a, b, c;
        begin
            if ((a == 0) && (b == 0) && (c == 0))
                pick_headroom = NONE;
            else if ((a >= b) && (a >= c))
                pick_headroom = S1;
            else if (b >= c)
                pick_headroom = S2;
            else
                pick_headroom = S3;
        end
    endfunction

    function [1:0] pick_coolest;
        input [7:0] t1, t2, t3;
        input e1, e2, e3;
        begin
            if (!e1 && !e2 && !e3)
                pick_coolest = NONE;
            else if (e1 && (!e2 || (t1 <= t2)) && (!e3 || (t1 <= t3)))
                pick_coolest = S1;
            else if (e2 && (!e3 || (t2 <= t3)))
                pick_coolest = S2;
            else
                pick_coolest = S3;
        end
    endfunction

    function [1:0] pick_fixed_priority;
        input [1:0] mode;
        input e1, e2, e3;
        begin
            if (!e1 && !e2 && !e3)
                pick_fixed_priority = NONE;
            else if (mode == 2'b10) begin
                if (e1) pick_fixed_priority = S1;
                else if (e2) pick_fixed_priority = S2;
                else pick_fixed_priority = S3;
            end else begin
                if (e3) pick_fixed_priority = S3;
                else if (e2) pick_fixed_priority = S2;
                else pick_fixed_priority = S1;
            end
        end
    endfunction

    assign best_receiver =
        (mode_select == 2'b00) ? pick_headroom(s1_headroom_score, s2_headroom_score, s3_headroom_score) :
        (mode_select == 2'b01) ? pick_coolest(s1_pred_temp, s2_pred_temp, s3_pred_temp,
                                               s1_receiver_eligible, s2_receiver_eligible, s3_receiver_eligible) :
                                 pick_fixed_priority(mode_select, s1_receiver_eligible,
                                                     s2_receiver_eligible, s3_receiver_eligible);
endmodule

Writing priority_eligibility.v


In [11]:
%%writefile redistribution_manager.v
module redistribution_manager (
    input  [15:0] s1_pred_power, s2_pred_power, s3_pred_power,
    input  [7:0]  s1_pred_temp, s2_pred_temp, s3_pred_temp,

    input         s1_pred_high, s2_pred_high, s3_pred_high,
    input         s1_thermal_redist_trigger,
    input         s2_thermal_redist_trigger,
    input         s3_thermal_redist_trigger,

    input  [1:0]  best_receiver,

    output [1:0]  source_server,
    output [1:0]  receiver_server,
    output        redistribution_enable,
    output [15:0] redistribution_amount,
    output        emergency_power_restriction_trigger,
    output        cooling_boost_required
);
    localparam [1:0] NONE = 2'b00;
    localparam [1:0] S1   = 2'b01;
    localparam [1:0] S2   = 2'b10;
    localparam [1:0] S3   = 2'b11;

    localparam [15:0] SOURCE_THRESHOLD     = 16'd750;
    localparam [15:0] RECEIVER_TARGET      = 16'd750;
    localparam [15:0] MIN_SOURCE_POWER     = 16'd400;
    localparam [7:0]  COOLING_TEMP_LIMIT   = 8'd70;

    wire s1_source_trigger = s1_pred_high || s1_thermal_redist_trigger;
    wire s2_source_trigger = s2_pred_high || s2_thermal_redist_trigger;
    wire s3_source_trigger = s3_pred_high || s3_thermal_redist_trigger;

    wire [16:0] s1_severity = s1_source_trigger ?
        ({1'b0, s1_pred_power} + ({9'd0, s1_pred_temp} * 17'd7)) : 17'd0;
    wire [16:0] s2_severity = s2_source_trigger ?
        ({1'b0, s2_pred_power} + ({9'd0, s2_pred_temp} * 17'd7)) : 17'd0;
    wire [16:0] s3_severity = s3_source_trigger ?
        ({1'b0, s3_pred_power} + ({9'd0, s3_pred_temp} * 17'd7)) : 17'd0;

    function [1:0] pick_worst;
        input [16:0] score1, score2, score3;
        begin
            if ((score1 == 0) && (score2 == 0) && (score3 == 0))
                pick_worst = NONE;
            else if ((score1 >= score2) && (score1 >= score3))
                pick_worst = S1;
            else if (score2 >= score3)
                pick_worst = S2;
            else
                pick_worst = S3;
        end
    endfunction

    assign source_server = pick_worst(s1_severity, s2_severity, s3_severity);
    assign receiver_server = best_receiver;

    wire [15:0] source_power =
        (source_server == S1) ? s1_pred_power :
        (source_server == S2) ? s2_pred_power :
        (source_server == S3) ? s3_pred_power : 16'd0;

    wire [7:0] source_temp =
        (source_server == S1) ? s1_pred_temp :
        (source_server == S2) ? s2_pred_temp :
        (source_server == S3) ? s3_pred_temp : 8'd0;

    wire [15:0] receiver_power =
        (receiver_server == S1) ? s1_pred_power :
        (receiver_server == S2) ? s2_pred_power :
        (receiver_server == S3) ? s3_pred_power : 16'd0;

    assign cooling_boost_required =
        (source_server != NONE) &&
        (source_temp >= COOLING_TEMP_LIMIT);

    // Source is reduced toward 750 W, not 700 W.
    wire [15:0] needed_reduction =
        (source_power > SOURCE_THRESHOLD) ?
        (source_power - SOURCE_THRESHOLD) : 16'd0;

    // Receiver must have been selected upstream only if it was <600 W.
    // Once selected, do not drive it beyond 750 W.
    wire [15:0] receiver_headroom =
        (receiver_power < RECEIVER_TARGET) ?
        (RECEIVER_TARGET - receiver_power) : 16'd0;

    // Source cannot be reduced below 400 W.
    wire [15:0] max_allowed_by_floor =
        (source_power > MIN_SOURCE_POWER) ?
        (source_power - MIN_SOURCE_POWER) : 16'd0;

    wire [15:0] amount_1 =
        (needed_reduction < receiver_headroom) ?
        needed_reduction : receiver_headroom;

    wire [15:0] final_amount =
        (amount_1 < max_allowed_by_floor) ?
        amount_1 : max_allowed_by_floor;

    assign redistribution_enable =
        (source_server != NONE) &&
        (best_receiver != NONE) &&
        (source_server != best_receiver) &&
        (final_amount != 16'd0);

    // No receiver -> do NOT force an unsafe transfer.
    assign emergency_power_restriction_trigger =
        (source_server != NONE) &&
        (best_receiver == NONE) &&
        ((source_power > SOURCE_THRESHOLD) ||
         (source_temp >= COOLING_TEMP_LIMIT));

    assign redistribution_amount =
        redistribution_enable ? final_amount : 16'd0;
endmodule

Writing redistribution_manager.v


In [12]:
%%writefile power_allocation_engine.v
module power_allocation_engine (
    input clk,
    input rst,

    input [15:0] s1_pred_power, s2_pred_power, s3_pred_power,
    input [15:0] s1_pred_it, s2_pred_it, s3_pred_it,
    input [15:0] s1_pred_cooling, s2_pred_cooling, s3_pred_cooling,
    input [15:0] s1_pred_hvac, s2_pred_hvac, s3_pred_hvac,

    input        s1_reserve_trigger, s2_reserve_trigger, s3_reserve_trigger,
    input [15:0] s1_reserve_injection, s2_reserve_injection, s3_reserve_injection,

    input [1:0] source_server,
    input [1:0] receiver_server,
    input       redistribution_enable,
    input [15:0] redistribution_amount,
    input       emergency_power_restriction_trigger,
    input       cooling_boost_required,

    output reg [15:0] s1_allocated, s2_allocated, s3_allocated,
    output reg [15:0] s1_extra_cooling, s2_extra_cooling, s3_extra_cooling,
    output reg        full_cooling_engage
);
    localparam [15:0] RAMP_STEP          = 16'd15;
    localparam [15:0] RESTRICTION_TARGET = 16'd750;
    localparam [15:0] MAX_POWER          = 16'd1000;
    localparam [1:0]  NONE = 2'b00, S1 = 2'b01, S2 = 2'b10, S3 = 2'b11;

    reg [15:0] s1_target, s2_target, s3_target;
    reg s1_is_restriction, s2_is_restriction, s3_is_restriction;

    always @(*) begin
        s1_target = s1_pred_power;
        s2_target = s2_pred_power;
        s3_target = s3_pred_power;
        s1_is_restriction = 1'b0;
        s2_is_restriction = 1'b0;
        s3_is_restriction = 1'b0;

        if (source_server == S1 && redistribution_enable)
            s1_target = (redistribution_amount >= s1_pred_power) ? 16'd0 : (s1_pred_power - redistribution_amount);
        else if (receiver_server == S1 && redistribution_enable)
            s1_target = s1_pred_power + redistribution_amount;
        else if (source_server == S1 && emergency_power_restriction_trigger) begin
            s1_target = (s1_pred_power > RESTRICTION_TARGET) ? RESTRICTION_TARGET : s1_pred_power;
            s1_is_restriction = (s1_pred_power > RESTRICTION_TARGET);
        end else if (s1_reserve_trigger)
            s1_target = s1_pred_power + s1_reserve_injection;

        if (source_server == S2 && redistribution_enable)
            s2_target = (redistribution_amount >= s2_pred_power) ? 16'd0 : (s2_pred_power - redistribution_amount);
        else if (receiver_server == S2 && redistribution_enable)
            s2_target = s2_pred_power + redistribution_amount;
        else if (source_server == S2 && emergency_power_restriction_trigger) begin
            s2_target = (s2_pred_power > RESTRICTION_TARGET) ? RESTRICTION_TARGET : s2_pred_power;
            s2_is_restriction = (s2_pred_power > RESTRICTION_TARGET);
        end else if (s2_reserve_trigger)
            s2_target = s2_pred_power + s2_reserve_injection;

        if (source_server == S3 && redistribution_enable)
            s3_target = (redistribution_amount >= s3_pred_power) ? 16'd0 : (s3_pred_power - redistribution_amount);
        else if (receiver_server == S3 && redistribution_enable)
            s3_target = s3_pred_power + redistribution_amount;
        else if (source_server == S3 && emergency_power_restriction_trigger) begin
            s3_target = (s3_pred_power > RESTRICTION_TARGET) ? RESTRICTION_TARGET : s3_pred_power;
            s3_is_restriction = (s3_pred_power > RESTRICTION_TARGET);
        end else if (s3_reserve_trigger)
            s3_target = s3_pred_power + s3_reserve_injection;

        // Clamp all targets to the project maximum.
        if (s1_target > MAX_POWER) s1_target = MAX_POWER;
        if (s2_target > MAX_POWER) s2_target = MAX_POWER;
        if (s3_target > MAX_POWER) s3_target = MAX_POWER;
    end

    function [15:0] calc_extra_cooling;
        input [15:0] pred_it, pred_cooling, pred_hvac;
        reg [31:0] heat_term, required_total, current_total;
        begin
            // Simple deterministic cooling estimate for the simulation.
            heat_term = (70 * pred_it) / 1000;
            if (heat_term > 30) begin
                required_total = ((heat_term - 30) * 10) / 3;
                current_total = pred_cooling + pred_hvac;
                calc_extra_cooling =
                    (required_total > current_total) ?
                    (required_total - current_total) : 16'd0;
            end else begin
                calc_extra_cooling = 16'd0;
            end
        end
    endfunction

    wire [15:0] s1_cool_calc = calc_extra_cooling(s1_pred_it, s1_pred_cooling, s1_pred_hvac);
    wire [15:0] s2_cool_calc = calc_extra_cooling(s2_pred_it, s2_pred_cooling, s2_pred_hvac);
    wire [15:0] s3_cool_calc = calc_extra_cooling(s3_pred_it, s3_pred_cooling, s3_pred_hvac);

    always @(posedge clk or posedge rst) begin
        if (rst) begin
            s1_allocated <= 16'd0;
            s2_allocated <= 16'd0;
            s3_allocated <= 16'd0;
            s1_extra_cooling <= 16'd0;
            s2_extra_cooling <= 16'd0;
            s3_extra_cooling <= 16'd0;
            full_cooling_engage <= 1'b0;
        end else begin
            if (s1_target > s1_allocated)
                s1_allocated <= ((s1_target - s1_allocated) > RAMP_STEP) ? (s1_allocated + RAMP_STEP) : s1_target;
            else if (s1_target < s1_allocated)
                s1_allocated <= ((s1_allocated - s1_target) > RAMP_STEP) ? (s1_allocated - RAMP_STEP) : s1_target;

            if (s2_target > s2_allocated)
                s2_allocated <= ((s2_target - s2_allocated) > RAMP_STEP) ? (s2_allocated + RAMP_STEP) : s2_target;
            else if (s2_target < s2_allocated)
                s2_allocated <= ((s2_allocated - s2_target) > RAMP_STEP) ? (s2_allocated - RAMP_STEP) : s2_target;

            if (s3_target > s3_allocated)
                s3_allocated <= ((s3_target - s3_allocated) > RAMP_STEP) ? (s3_allocated + RAMP_STEP) : s3_target;
            else if (s3_target < s3_allocated)
                s3_allocated <= ((s3_allocated - s3_target) > RAMP_STEP) ? (s3_allocated - RAMP_STEP) : s3_target;

            s1_extra_cooling <= (source_server == S1 && cooling_boost_required) ? s1_cool_calc : 16'd0;
            s2_extra_cooling <= (source_server == S2 && cooling_boost_required) ? s2_cool_calc : 16'd0;
            s3_extra_cooling <= (source_server == S3 && cooling_boost_required) ? s3_cool_calc : 16'd0;

            full_cooling_engage <= s1_is_restriction || s2_is_restriction || s3_is_restriction;
        end
    end
endmodule

Writing power_allocation_engine.v


In [13]:
%%writefile safety_check.v
module safety_check (
    input [15:0] s1_allocated, s2_allocated, s3_allocated,
    input [15:0] reserve_charge_level,
    input [15:0] s1_reserve_injection, s2_reserve_injection, s3_reserve_injection,
    input        s1_reserve_trigger, s2_reserve_trigger, s3_reserve_trigger,
    input        full_cooling_engage,
    input [1:0]  source_server, receiver_server,
    input        redistribution_enable,
    input        emergency_power_restriction_trigger,

    output       s1_allocation_valid, s2_allocation_valid, s3_allocation_valid,
    output       reserve_valid,
    output       reserve_injection_consistent,
    output       redistribution_consistent,
    output       restriction_consistent,
    output       final_valid
);
    localparam [1:0] NONE = 2'b00;
    localparam [15:0] MAX_POWER   = 16'd1000;
    localparam [15:0] RESERVE_MAX = 16'd10000;

    assign s1_allocation_valid = (s1_allocated <= MAX_POWER);
    assign s2_allocation_valid = (s2_allocated <= MAX_POWER);
    assign s3_allocation_valid = (s3_allocated <= MAX_POWER);

    assign reserve_valid = (reserve_charge_level <= RESERVE_MAX);

    assign reserve_injection_consistent =
        ((s1_reserve_trigger == 1'b1) || (s1_reserve_injection == 16'd0)) &&
        ((s2_reserve_trigger == 1'b1) || (s2_reserve_injection == 16'd0)) &&
        ((s3_reserve_trigger == 1'b1) || (s3_reserve_injection == 16'd0));

    assign redistribution_consistent =
        (!redistribution_enable) ||
        ((source_server != NONE) &&
         (receiver_server != NONE) &&
         (source_server != receiver_server));

    assign restriction_consistent =
        (!emergency_power_restriction_trigger) || full_cooling_engage;

    assign final_valid =
        s1_allocation_valid &&
        s2_allocation_valid &&
        s3_allocation_valid &&
        reserve_valid &&
        reserve_injection_consistent &&
        redistribution_consistent &&
        restriction_consistent;
endmodule

Writing safety_check.v


In [14]:
%%writefile green_core_controller.v
module green_core_controller (
    input clk,
    input rst,

    input [15:0] s1_pred_it, s1_pred_cooling, s1_pred_hvac, s1_pred_pump,
    input [15:0] s2_pred_it, s2_pred_cooling, s2_pred_hvac, s2_pred_pump,
    input [15:0] s3_pred_it, s3_pred_cooling, s3_pred_hvac, s3_pred_pump,

    input [7:0]  s1_pred_temp, s2_pred_temp, s3_pred_temp,
    input [1:0]  s1_maint, s2_maint, s3_maint,
    input [1:0]  mode_select,

    output [15:0] s1_pred_total, s2_pred_total, s3_pred_total,
    output [2:0]  s1_pred_zone, s2_pred_zone, s3_pred_zone,
    output        s1_pred_high, s2_pred_high, s3_pred_high,

    output [7:0]  s1_effective_temp, s2_effective_temp, s3_effective_temp,
    output        s1_health_eligible, s2_health_eligible, s3_health_eligible,
    output        s1_receiver_eligible, s2_receiver_eligible, s3_receiver_eligible,

    output [1:0]  best_receiver,
    output [1:0]  source_server, receiver_server,
    output        redistribution_enable,
    output [15:0] redistribution_amount,
    output        emergency_power_restriction_trigger,
    output        cooling_boost_required,

    output [15:0] s1_reserve_injection, s2_reserve_injection, s3_reserve_injection,
    output [15:0] reserve_cooling_injection,
    output [15:0] reserve_charge_level,

    output [15:0] s1_allocated, s2_allocated, s3_allocated,
    output [15:0] s1_extra_cooling, s2_extra_cooling, s3_extra_cooling,
    output        full_cooling_engage,

    output        s1_allocation_valid, s2_allocation_valid, s3_allocation_valid,
    output        reserve_valid,
    output        reserve_injection_consistent,
    output        redistribution_consistent,
    output        restriction_consistent,
    output        final_valid
);

    wire s1_reserve_trigger, s2_reserve_trigger, s3_reserve_trigger;
    wire s1_cooling_boost_la, s2_cooling_boost_la, s3_cooling_boost_la;

    wire [1:0] s1_thermal_status, s2_thermal_status, s3_thermal_status;
    wire [1:0] s1_pred_thermal_status, s2_pred_thermal_status, s3_pred_thermal_status;
    wire s1_thermal_safe, s2_thermal_safe, s3_thermal_safe;
    wire s1_pred_thermal_safe, s2_pred_thermal_safe, s3_pred_thermal_safe;
    wire s1_thermal_redist_trigger, s2_thermal_redist_trigger, s3_thermal_redist_trigger;
    wire s1_cooling_malfunction, s2_cooling_malfunction, s3_cooling_malfunction;
    wire s1_maintenance_ok, s2_maintenance_ok, s3_maintenance_ok;

    load_analyzer_top LA (
        .s1_pred_it(s1_pred_it), .s1_pred_cooling(s1_pred_cooling), .s1_pred_hvac(s1_pred_hvac), .s1_pred_pump(s1_pred_pump),
        .s2_pred_it(s2_pred_it), .s2_pred_cooling(s2_pred_cooling), .s2_pred_hvac(s2_pred_hvac), .s2_pred_pump(s2_pred_pump),
        .s3_pred_it(s3_pred_it), .s3_pred_cooling(s3_pred_cooling), .s3_pred_hvac(s3_pred_hvac), .s3_pred_pump(s3_pred_pump),
        .s1_pred_total(s1_pred_total), .s2_pred_total(s2_pred_total), .s3_pred_total(s3_pred_total),
        .s1_pred_zone(s1_pred_zone), .s2_pred_zone(s2_pred_zone), .s3_pred_zone(s3_pred_zone),
        .s1_pred_high(s1_pred_high), .s2_pred_high(s2_pred_high), .s3_pred_high(s3_pred_high),
        .s1_reserve_trigger(s1_reserve_trigger), .s2_reserve_trigger(s2_reserve_trigger), .s3_reserve_trigger(s3_reserve_trigger),
        .s1_cooling_boost(s1_cooling_boost_la), .s2_cooling_boost(s2_cooling_boost_la), .s3_cooling_boost(s3_cooling_boost_la)
    );

    // Temporary wires for redistribution outputs, used by Health+Maintenance
    // to drive the reserve subsystem safely.
    wire [1:0] rm_source_server;
    wire [1:0] rm_receiver_server;
    wire rm_redistribution_enable;
    wire [15:0] rm_redistribution_amount;
    wire rm_emergency_restriction;
    wire rm_cooling_boost_required;

    wire [1:0] best_receiver_w;
    wire s1_receiver_eligible_w, s2_receiver_eligible_w, s3_receiver_eligible_w;

    health_maintenance_top HM (
        .clk(clk), .rst(rst),
        .s1_pred_it(s1_pred_it), .s2_pred_it(s2_pred_it), .s3_pred_it(s3_pred_it),
        .s1_pred_cooling(s1_pred_cooling), .s2_pred_cooling(s2_pred_cooling), .s3_pred_cooling(s3_pred_cooling),
        .s1_pred_hvac(s1_pred_hvac), .s2_pred_hvac(s2_pred_hvac), .s3_pred_hvac(s3_pred_hvac),
        .s1_pred_temp(s1_pred_temp), .s2_pred_temp(s2_pred_temp), .s3_pred_temp(s3_pred_temp),
        .s1_maint(s1_maint), .s2_maint(s2_maint), .s3_maint(s3_maint),
        .s1_power(s1_pred_total), .s2_power(s2_pred_total), .s3_power(s3_pred_total),
        .s1_reserve_trigger_la(s1_reserve_trigger), .s2_reserve_trigger_la(s2_reserve_trigger), .s3_reserve_trigger_la(s3_reserve_trigger),
        .s1_cooling_boost_la(s1_cooling_boost_la), .s2_cooling_boost_la(s2_cooling_boost_la), .s3_cooling_boost_la(s3_cooling_boost_la),
        .source_server(rm_source_server),
        .emergency_power_restriction_trigger(rm_emergency_restriction),
        .s1_effective_temp(s1_effective_temp), .s2_effective_temp(s2_effective_temp), .s3_effective_temp(s3_effective_temp),
        .s1_thermal_status(s1_thermal_status), .s2_thermal_status(s2_thermal_status), .s3_thermal_status(s3_thermal_status),
        .s1_pred_thermal_status(s1_pred_thermal_status), .s2_pred_thermal_status(s2_pred_thermal_status), .s3_pred_thermal_status(s3_pred_thermal_status),
        .s1_thermal_safe(s1_thermal_safe), .s2_thermal_safe(s2_thermal_safe), .s3_thermal_safe(s3_thermal_safe),
        .s1_pred_thermal_safe(s1_pred_thermal_safe), .s2_pred_thermal_safe(s2_pred_thermal_safe), .s3_pred_thermal_safe(s3_pred_thermal_safe),
        .s1_thermal_redist_trigger(s1_thermal_redist_trigger), .s2_thermal_redist_trigger(s2_thermal_redist_trigger), .s3_thermal_redist_trigger(s3_thermal_redist_trigger),
        .s1_cooling_malfunction(s1_cooling_malfunction), .s2_cooling_malfunction(s2_cooling_malfunction), .s3_cooling_malfunction(s3_cooling_malfunction),
        .s1_maintenance_ok(s1_maintenance_ok), .s2_maintenance_ok(s2_maintenance_ok), .s3_maintenance_ok(s3_maintenance_ok),
        .s1_health_eligible(s1_health_eligible), .s2_health_eligible(s2_health_eligible), .s3_health_eligible(s3_health_eligible),
        .reserve_cooling_injection(reserve_cooling_injection),
        .reserve_charge_level(reserve_charge_level),
        .s1_reserve_injection(s1_reserve_injection), .s2_reserve_injection(s2_reserve_injection), .s3_reserve_injection(s3_reserve_injection),
        .cooling_reserve_active()
    );

    priority_eligibility PE (
        .s1_pred_power(s1_pred_total), .s2_pred_power(s2_pred_total), .s3_pred_power(s3_pred_total),
        .s1_health_eligible(s1_health_eligible), .s2_health_eligible(s2_health_eligible), .s3_health_eligible(s3_health_eligible),
        .s1_pred_temp(s1_pred_temp), .s2_pred_temp(s2_pred_temp), .s3_pred_temp(s3_pred_temp),
        .mode_select(mode_select),
        .s1_receiver_eligible(s1_receiver_eligible_w), .s2_receiver_eligible(s2_receiver_eligible_w), .s3_receiver_eligible(s3_receiver_eligible_w),
        .best_receiver(best_receiver_w)
    );

    assign s1_receiver_eligible = s1_receiver_eligible_w;
    assign s2_receiver_eligible = s2_receiver_eligible_w;
    assign s3_receiver_eligible = s3_receiver_eligible_w;
    assign best_receiver = best_receiver_w;

    redistribution_manager RM (
        .s1_pred_power(s1_pred_total), .s2_pred_power(s2_pred_total), .s3_pred_power(s3_pred_total),
        .s1_pred_temp(s1_pred_temp), .s2_pred_temp(s2_pred_temp), .s3_pred_temp(s3_pred_temp),
        .s1_pred_high(s1_pred_high), .s2_pred_high(s2_pred_high), .s3_pred_high(s3_pred_high),
        .s1_thermal_redist_trigger(s1_thermal_redist_trigger),
        .s2_thermal_redist_trigger(s2_thermal_redist_trigger),
        .s3_thermal_redist_trigger(s3_thermal_redist_trigger),
        .best_receiver(best_receiver_w),
        .source_server(rm_source_server), .receiver_server(rm_receiver_server),
        .redistribution_enable(rm_redistribution_enable),
        .redistribution_amount(rm_redistribution_amount),
        .emergency_power_restriction_trigger(rm_emergency_restriction),
        .cooling_boost_required(rm_cooling_boost_required)
    );

    assign source_server = rm_source_server;
    assign receiver_server = rm_receiver_server;
    assign redistribution_enable = rm_redistribution_enable;
    assign redistribution_amount = rm_redistribution_amount;
    assign emergency_power_restriction_trigger = rm_emergency_restriction;
    assign cooling_boost_required = rm_cooling_boost_required;

    power_allocation_engine ALLOC (
        .clk(clk), .rst(rst),
        .s1_pred_power(s1_pred_total), .s2_pred_power(s2_pred_total), .s3_pred_power(s3_pred_total),
        .s1_pred_it(s1_pred_it), .s2_pred_it(s2_pred_it), .s3_pred_it(s3_pred_it),
        .s1_pred_cooling(s1_pred_cooling), .s2_pred_cooling(s2_pred_cooling), .s3_pred_cooling(s3_pred_cooling),
        .s1_pred_hvac(s1_pred_hvac), .s2_pred_hvac(s2_pred_hvac), .s3_pred_hvac(s3_pred_hvac),
        .s1_reserve_trigger(s1_reserve_trigger), .s2_reserve_trigger(s2_reserve_trigger), .s3_reserve_trigger(s3_reserve_trigger),
        .s1_reserve_injection(s1_reserve_injection), .s2_reserve_injection(s2_reserve_injection), .s3_reserve_injection(s3_reserve_injection),
        .source_server(rm_source_server), .receiver_server(rm_receiver_server),
        .redistribution_enable(rm_redistribution_enable),
        .redistribution_amount(rm_redistribution_amount),
        .emergency_power_restriction_trigger(rm_emergency_restriction),
        .cooling_boost_required(rm_cooling_boost_required),
        .s1_allocated(s1_allocated), .s2_allocated(s2_allocated), .s3_allocated(s3_allocated),
        .s1_extra_cooling(s1_extra_cooling), .s2_extra_cooling(s2_extra_cooling), .s3_extra_cooling(s3_extra_cooling),
        .full_cooling_engage(full_cooling_engage)
    );

    safety_check SAFETY (
        .s1_allocated(s1_allocated), .s2_allocated(s2_allocated), .s3_allocated(s3_allocated),
        .reserve_charge_level(reserve_charge_level),
        .s1_reserve_injection(s1_reserve_injection), .s2_reserve_injection(s2_reserve_injection), .s3_reserve_injection(s3_reserve_injection),
        .s1_reserve_trigger(s1_reserve_trigger), .s2_reserve_trigger(s2_reserve_trigger), .s3_reserve_trigger(s3_reserve_trigger),
        .full_cooling_engage(full_cooling_engage),
        .source_server(rm_source_server), .receiver_server(rm_receiver_server),
        .redistribution_enable(rm_redistribution_enable),
        .emergency_power_restriction_trigger(rm_emergency_restriction),
        .s1_allocation_valid(s1_allocation_valid), .s2_allocation_valid(s2_allocation_valid), .s3_allocation_valid(s3_allocation_valid),
        .reserve_valid(reserve_valid),
        .reserve_injection_consistent(reserve_injection_consistent),
        .redistribution_consistent(redistribution_consistent),
        .restriction_consistent(restriction_consistent),
        .final_valid(final_valid)
    );
endmodule

Writing green_core_controller.v


In [21]:
%%writefile system_frame_tb.v
module system_frame_tb;
    reg clk = 1'b0; // Initialize clk here
    reg rst;

    // Current data: reference only.
    reg [15:0] s1_it, s1_cooling, s1_hvac, s1_pump;
    reg [15:0] s2_it, s2_cooling, s2_hvac, s2_pump;
    reg [15:0] s3_it, s3_cooling, s3_hvac, s3_pump;

    // Predicted data: functional FPGA inputs.
    reg [15:0] s1_pred_it, s1_pred_cooling, s1_pred_hvac, s1_pred_pump;
    reg [15:0] s2_pred_it, s2_pred_cooling, s2_pred_hvac, s2_pred_pump;
    reg [15:0] s3_pred_it, s3_pred_cooling, s3_pred_hvac, s3_pred_pump;

    reg [7:0] s1_temp_ref, s2_temp_ref, s3_temp_ref;
    reg [7:0] s1_pred_temp, s2_pred_temp, s3_pred_temp;
    reg [1:0] s1_maint, s2_maint, s3_maint;
    reg [1:0] mode_select;

    wire [15:0] ref_s1_current_total = s1_it + s1_cooling + s1_hvac + s1_pump;
    wire [15:0] ref_s2_current_total = s2_it + s2_cooling + s2_hvac + s2_pump;
    wire [15:0] ref_s3_current_total = s3_it + s3_cooling + s3_hvac + s3_pump;

    wire [15:0] s1_pred_total, s2_pred_total, s3_pred_total;
    wire [2:0] s1_pred_zone, s2_pred_zone, s3_pred_zone;
    wire s1_pred_high, s2_pred_high, s3_pred_high;

    wire [7:0] s1_effective_temp, s2_effective_temp, s3_effective_temp;
    wire s1_health_eligible, s2_health_eligible, s3_health_eligible;
    wire s1_receiver_eligible, s2_receiver_eligible, s3_receiver_eligible;
    wire [1:0] best_receiver;

    wire [1:0] source_server, receiver_server;
    wire redistribution_enable;
    wire [15:0] redistribution_amount;
    wire emergency_power_restriction_trigger;
    wire cooling_boost_required;

    wire [15:0] s1_reserve_injection, s2_reserve_injection, s3_reserve_injection;
    wire [15:0] reserve_cooling_injection, reserve_charge_level;
    wire [15:0] s1_allocated, s2_allocated, s3_allocated;
    wire [15:0] s1_extra_cooling, s2_extra_cooling, s3_extra_cooling;
    wire full_cooling_engage;

    wire s1_allocation_valid, s2_allocation_valid, s3_allocation_valid;
    wire reserve_valid, reserve_injection_consistent;
    wire redistribution_consistent, restriction_consistent, final_valid;

    green_core_controller DUT (
        .clk(clk), .rst(rst),

        .s1_pred_it(s1_pred_it), .s1_pred_cooling(s1_pred_cooling), .s1_pred_hvac(s1_pred_hvac), .s1_pred_pump(s1_pred_pump),
        .s2_pred_it(s2_pred_it), .s2_pred_cooling(s2_pred_cooling), .s2_pred_hvac(s2_pred_hvac), .s2_pred_pump(s2_pred_pump),
        .s3_pred_it(s3_pred_it), .s3_pred_cooling(s3_pred_cooling), .s3_pred_hvac(s3_pred_hvac), .s3_pred_pump(s3_pred_pump),

        .s1_pred_temp(s1_pred_temp), .s2_pred_temp(s2_pred_temp), .s3_pred_temp(s3_pred_temp),
        .s1_maint(s1_maint), .s2_maint(s2_maint), .s3_maint(s3_maint),
        .mode_select(mode_select),

        .s1_pred_total(s1_pred_total), .s2_pred_total(s2_pred_total), .s3_pred_total(s3_pred_total),
        .s1_pred_zone(s1_pred_zone), .s2_pred_zone(s2_pred_zone), .s3_pred_zone(s3_pred_zone),
        .s1_pred_high(s1_pred_high), .s2_pred_high(s2_pred_high), .s3_pred_high(s3_pred_high),

        .s1_effective_temp(s1_effective_temp), .s2_effective_temp(s2_effective_temp), .s3_effective_temp(s3_effective_temp),
        .s1_health_eligible(s1_health_eligible), .s2_health_eligible(s2_health_eligible), .s3_health_eligible(s3_health_eligible),
        .s1_receiver_eligible(s1_receiver_eligible), .s2_receiver_eligible(s2_receiver_eligible), .s3_receiver_eligible(s3_receiver_eligible),
        .best_receiver(best_receiver),

        .source_server(source_server), .receiver_server(receiver_server),
        .redistribution_enable(redistribution_enable), .redistribution_amount(redistribution_amount),
        .emergency_power_restriction_trigger(emergency_power_restriction_trigger),
        .cooling_boost_required(cooling_boost_required),

        .s1_reserve_injection(s1_reserve_injection), .s2_reserve_injection(s2_reserve_injection), .s3_reserve_injection(s3_reserve_injection),
        .reserve_cooling_injection(reserve_cooling_injection), .reserve_charge_level(reserve_charge_level),

        .s1_allocated(s1_allocated), .s2_allocated(s2_allocated), .s3_allocated(s3_allocated),
        .s1_extra_cooling(s1_extra_cooling), .s2_extra_cooling(s2_extra_cooling), .s3_extra_cooling(s3_extra_cooling),
        .full_cooling_engage(full_cooling_engage),

        .s1_allocation_valid(s1_allocation_valid), .s2_allocation_valid(s2_allocation_valid), .s3_allocation_valid(s3_allocation_valid),
        .reserve_valid(reserve_valid), .reserve_injection_consistent(reserve_injection_consistent),
        .redistribution_consistent(redistribution_consistent), .restriction_consistent(restriction_consistent),
        .final_valid(final_valid)
    );

    always #5 clk = ~clk;

    integer fd, status, frame_num;
    integer result_fd;   // NEW: CSV output for graphing in Colab/Python

    initial begin
        $dumpfile("system_run.vcd");
        $dumpvars(0, system_frame_tb);

        // Safe initial conditions.
        // clk = 1'b0; // Initialize clk here - REMOVED
        rst = 1'b1;
        s1_it=0; s1_cooling=0; s1_hvac=0; s1_pump=0;
        s2_it=0; s2_cooling=0; s2_hvac=0; s2_pump=0;
        s3_it=0; s3_cooling=0; s3_hvac=0; s3_pump=0;
        s1_pred_it=0; s1_pred_cooling=0; s1_pred_hvac=0; s1_pred_pump=0;
        s2_pred_it=0; s2_pred_cooling=0; s2_pred_hvac=0; s2_pred_pump=0;
        s3_pred_it=0; s3_pred_cooling=0; s3_pred_hvac=0; s3_pred_pump=0;
        s1_temp_ref=0; s2_temp_ref=0; s3_temp_ref=0;
        s1_pred_temp=0; s2_pred_temp=0; s3_pred_temp=0;
        s1_maint=0; s2_maint=0; s3_maint=0;
        mode_select=0;

        #12;
        rst = 1'b0;

        fd = $fopen("system_frames.txt", "r");
        if (fd == 0) begin
            $display("ERROR: could not open system_frames.txt");
            $finish;
        end

        // NEW: machine-readable per-frame log for graphing in Colab/Python.
        // One row per simulated frame, covering every signal the six
        // requested graphs need (allocation, temperature before/after,
        // predicted-vs-current power, reserve charge, redistribution).
        result_fd = $fopen("system_results.csv", "w");
        if (result_fd == 0) begin
            $display("ERROR: could not open system_results.csv for writing");
            $finish;
        end
        $fwrite(result_fd,
            {"frame,time,current_s1,pred_s1,current_s2,pred_s2,current_s3,pred_s3,"
            "alloc_s1,alloc_s2,alloc_s3,source,receiver,redist_enable,redist_amount,"
            "pred_temp_s1,pred_temp_s2,pred_temp_s3,eff_temp_s1,eff_temp_s2,eff_temp_s3,"
            "reserve_charge,final_valid\n"});

        frame_num = 0;

        while (!$feof(fd)) begin
            status = $fscanf(fd,
                "%d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d %d\n",

                s1_it, s1_cooling, s1_hvac, s1_pump,
                s1_pred_it, s1_pred_cooling, s1_pred_hvac, s1_pred_pump,
                s1_temp_ref, s1_pred_temp, s1_maint,

                s2_it, s2_cooling, s2_hvac, s2_pump,
                s2_pred_it, s2_pred_cooling, s2_pred_hvac, s2_pred_pump,
                s2_temp_ref, s2_pred_temp, s2_maint,

                s3_it, s3_cooling, s3_hvac, s3_pump,
                s3_pred_it, s3_pred_cooling, s3_pred_hvac, s3_pred_pump,
                s3_temp_ref, s3_pred_temp, s3_maint,

                mode_select
            );

            if (status == 34) begin
                #10;
                $display("========================================================");
                $display("FRAME %0d | TIME=%0t ns | MODE=%0d", frame_num, $time, mode_select);
                $display("REF  S1=%0d W | PRED S1=%0d W | Zone=%0d | PredHigh=%b", ref_s1_current_total, s1_pred_total, s1_pred_zone, s1_pred_high);
                $display("REF  S2=%0d W | PRED S2=%0d W | Zone=%0d | PredHigh=%b", ref_s2_current_total, s2_pred_total, s2_pred_zone, s2_pred_high);
                $display("REF  S3=%0d W | PRED S3=%0d W | Zone=%0d | PredHigh=%b", ref_s3_current_total, s3_pred_total, s3_pred_zone, s3_pred_high);
                $display("TEMP S1=%0dC S2=%0dC S3=%0dC | HEALTH=%b%b%b", s1_effective_temp, s2_effective_temp, s3_effective_temp, s1_health_eligible, s2_health_eligible, s3_health_eligible);
                $display("RECV S1=%b S2=%b S3=%b | BEST=%0d", s1_receiver_eligible, s2_receiver_eligible, s3_receiver_eligible, best_receiver);
                $display("REDIST SOURCE=%0d RECEIVER=%0d ENABLE=%b AMOUNT=%0d W", source_server, receiver_server, redistribution_enable, redistribution_amount);
                $display("ALLOC S1=%0d W S2=%0d W S3=%0d W | COOLING=%b EXTRA=%0d/%0d/%0d", s1_allocated, s2_allocated, s3_allocated, cooling_boost_required, s1_extra_cooling, s2_extra_cooling, s3_extra_cooling);
                $display("SAFETY final=%b alloc=%b%b%b reserve=%b redist=%b restriction=%b", final_valid, s1_allocation_valid, s2_allocation_valid, s3_allocation_valid, reserve_valid, redistribution_consistent, restriction_consistent);
                $display("========================================================");

                // NEW: one CSV row per frame, same data as the $display above
                // but machine-readable for pandas/matplotlib in Colab.
                $fwrite(result_fd,
                    "%0d,%0t,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d\n",
                    frame_num, $time,
                    ref_s1_current_total, s1_pred_total,
                    ref_s2_current_total, s2_pred_total,
                    ref_s3_current_total, s3_pred_total,
                    s1_allocated, s2_allocated, s3_allocated,
                    source_server, receiver_server, redistribution_enable, redistribution_amount,
                    s1_pred_temp, s2_pred_temp, s3_pred_temp,
                    s1_effective_temp, s2_effective_temp, s3_effective_temp,
                    reserve_charge_level,
                    final_valid
                );

                frame_num = frame_num + 1;
            end
        end

        $fclose(fd);
        $fclose(result_fd);
        $display("Simulation complete. Processed %0d frames.", frame_num);
        $display("Wrote system_results.csv (%0d rows) for graphing.", frame_num);
        $finish;
    end
endmodule

Overwriting system_frame_tb.v


In [23]:
import os
import subprocess
from pathlib import Path

iverilog = Path.home() / "iverilog" / "bin" / ("iverilog.exe" if os.name == "nt" else "iverilog")
if not iverilog.exists():
    iverilog = Path("iverilog")

# Icarus Verilog requires the CSV header to be one string literal.
testbench = Path("system_frame_tb.v")
lines = testbench.read_text().splitlines()
for index, line in enumerate(lines):
    if line.strip() == "$fwrite(result_fd," and index + 1 < len(lines) and "frame,time" in lines[index + 1]:
        lines[index:index + 5] = [
            '        $fwrite(result_fd, "frame,time,current_s1,pred_s1,current_s2,pred_s2,current_s3,pred_s3,alloc_s1,alloc_s2,alloc_s3,source,receiver,redist_enable,redist_amount,pred_temp_s1,pred_temp_s2,pred_temp_s3,eff_temp_s1,eff_temp_s2,eff_temp_s3,reserve_charge,final_valid\\n");'
        ]
        break
testbench.write_text("\n".join(lines) + "\n")

sources = [
    "health_maintenance.v",
    "health_maintenance_top.v",
    "load_analyzer.v",
    "load_analyzer_top.v",
    "green_core_controller.v",
    "power_allocation_engine.v",
    "priority_eligibility.v",
    "redistribution_manager.v",
    "reserve_power_supply.v",
    "safety_check.v",
    "system_frame_tb.v",
]
subprocess.run([str(iverilog), "-o", "sim", *sources], check=True)
print("Verilog compilation completed successfully.")

Verilog compilation completed successfully.


In [27]:
import os
import subprocess
from pathlib import Path

vvp = Path.home() / "iverilog" / "bin" / ("vvp.exe" if os.name == "nt" else "vvp")
if not vvp.exists():
    vvp = Path("vvp")

input_file = Path("system_frames.txt")
if not input_file.exists():
    raise FileNotFoundError(
        "system_frames.txt is required by system_frame_tb.v but is missing. "
        "Add the frame input data before running the simulation."
    )

result = subprocess.run([str(vvp), "sim"], capture_output=True, text=True, check=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
print("Simulation completed successfully.")

VCD info: dumpfile system_run.vcd opened for output.
FRAME 0 | TIME=22 ns | MODE=0
REF  S1=150 W | PRED S1=160 W | Zone=0 | PredHigh=0
REF  S2=260 W | PRED S2=270 W | Zone=0 | PredHigh=0
REF  S3=370 W | PRED S3=380 W | Zone=1 | PredHigh=0
TEMP S1=45C S2=44C S3=46C | HEALTH=111
RECV S1=1 S2=1 S3=1 | BEST=1
REDIST SOURCE=0 RECEIVER=1 ENABLE=0 AMOUNT=0 W
ALLOC S1=15 W S2=15 W S3=15 W | COOLING=0 EXTRA=0/0/0
SAFETY final=1 alloc=111 reserve=1 redist=1 restriction=1
FRAME 1 | TIME=32 ns | MODE=2
REF  S1=790 W | PRED S1=870 W | Zone=5 | PredHigh=1
REF  S2=260 W | PRED S2=270 W | Zone=0 | PredHigh=0
REF  S3=320 W | PRED S3=330 W | Zone=1 | PredHigh=0
TEMP S1=76C S2=55C S3=58C | HEALTH=111
RECV S1=0 S2=1 S3=1 | BEST=2
REDIST SOURCE=1 RECEIVER=2 ENABLE=1 AMOUNT=120 W
ALLOC S1=30 W S2=30 W S3=30 W | COOLING=1 EXTRA=10/0/0
SAFETY final=1 alloc=111 reserve=1 redist=1 restriction=1
FRAME 2 | TIME=42 ns | MODE=1
REF  S1=675 W | PRED S1=705 W | Zone=3 | PredHigh=0
REF  S2=350 W | PRED S2=360 W | Zone

In [ ]:
from pathlib import Path
import shutil

root = Path.cwd()
rtl_dir = root / "rtl"
simulation_dir = root / "simulation"
output_dir = simulation_dir / "output"
for directory in (rtl_dir, simulation_dir, output_dir):
    directory.mkdir(parents=True, exist_ok=True)

# Keep notebook-generated artifacts aligned with the checked-in project layout.
testbench = root / "system_frame_tb.v"
if testbench.exists():
    testbench_text = testbench.read_text()
    testbench_text = testbench_text.replace('"system_run.vcd"', '"simulation/output/system_run.vcd"')
    testbench_text = testbench_text.replace('"system_frames.txt"', '"simulation/system_frames.txt"')
    testbench_text = testbench_text.replace('"system_results.csv"', '"simulation/output/system_results.csv"')
    testbench.write_text(testbench_text)

def move_replace(source, destination):
    if source.exists():
        destination.unlink(missing_ok=True)
        shutil.move(str(source), str(destination))

for source in root.glob("*.v"):
    destination = simulation_dir / source.name if source.name == "system_frame_tb.v" else rtl_dir / source.name
    move_replace(source, destination)

move_replace(root / "system_frames.txt", simulation_dir / "system_frames.txt")
move_replace(root / "system_results.csv", output_dir / "system_results.csv")
move_replace(root / "system_run.vcd", output_dir / "system_run.vcd")
move_replace(root / "sim", output_dir / "sih_model.vvp")

print("Organized generated Verilog and simulation artifacts.")